# ECB FX Reference Rates Bronze Ingestion

Heavy notebook entrypoint for ingesting ECB daily euro foreign exchange reference rates into Bronze.

Execution modes:

- `backfill`: explicit inclusive date range
- `incremental`: rolling lookback window ending on the latest completed `Europe/Berlin` day

The target table must already exist. Run `00_platform_setup_catalog_schema.ipynb` first.

In [ ]:
import csv
import hashlib
import json
import time
import uuid
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from io import StringIO
from zoneinfo import ZoneInfo

import requests
from delta.tables import DeltaTable
from pyspark.sql import Row, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType, TimestampType

spark.conf.set("spark.sql.session.timeZone", "UTC")

ECB_LOCAL_TZ = ZoneInfo("Europe/Berlin")
ECB_API_BASE = "https://data-api.ecb.europa.eu/service/data/EXR"


def parse_iso_date(raw_value: str, field_name: str) -> date:
    try:
        return datetime.strptime(raw_value, "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(f"{field_name} must be in YYYY-MM-DD format, got: {raw_value}") from exc


def parse_quote_currencies(raw_value: str) -> list[str]:
    quote_currencies = []
    seen = set()
    for part in raw_value.split(","):
        currency = part.strip().upper()
        if not currency:
            continue
        if currency in seen:
            continue
        if len(currency) != 3 or not currency.isalpha():
            raise ValueError(f"Invalid currency code: {currency}")
        seen.add(currency)
        quote_currencies.append(currency)

    if not quote_currencies:
        raise ValueError("quote_currencies must contain at least one 3-letter currency code")

    return quote_currencies


def resolve_date_window(mode: str, start_date_raw: str, end_date_raw: str, lookback_days_raw: str) -> tuple[date, date]:
    normalized_mode = mode.strip().lower()
    if normalized_mode == "backfill":
        if not start_date_raw or not end_date_raw:
            raise ValueError("backfill mode requires start_date and end_date")

        start_date = parse_iso_date(start_date_raw, "start_date")
        end_date = parse_iso_date(end_date_raw, "end_date")
        if start_date > end_date:
            raise ValueError("start_date must be less than or equal to end_date")
        return start_date, end_date

    if normalized_mode != "incremental":
        raise ValueError(f"Unsupported mode: {mode}")

    lookback_days = int(lookback_days_raw or "7")
    if lookback_days < 1:
        raise ValueError("lookback_days must be greater than or equal to 1")

    local_today = datetime.now(ECB_LOCAL_TZ).date()
    end_date = local_today - timedelta(days=1)
    start_date = end_date - timedelta(days=lookback_days - 1)
    return start_date, end_date


def fetch_ecb_csv(quote_currencies: list[str], start_date: date, end_date: date, max_retries: int = 5) -> tuple[str, str, str]:
    series_key = f"D.{'+'.join(quote_currencies)}.EUR.SP00.A"
    request_url = f"{ECB_API_BASE}/{series_key}"
    params = {
        "startPeriod": start_date.isoformat(),
        "endPeriod": end_date.isoformat(),
        "format": "csvdata",
    }
    headers = {
        "Accept": "text/csv",
        "User-Agent": "market-macro-lakehouse/1.0",
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(request_url, params=params, headers=headers, timeout=30)
            if response.status_code == 200:
                return request_url, series_key, response.text
            if response.status_code == 429 or 500 <= response.status_code < 600:
                time.sleep((2 ** attempt) + 0.5)
                continue

            raise RuntimeError(f"ECB API error {response.status_code}: {response.text[:500]}")
        except requests.RequestException as exc:
            if attempt == max_retries - 1:
                raise RuntimeError("Failed to fetch ECB FX reference rates after retries") from exc
            time.sleep((2 ** attempt) + 0.5)

    raise RuntimeError("ECB API request exhausted retries unexpectedly")


def parse_ecb_rows(
    csv_text: str,
    quote_currencies: list[str],
    start_date: date,
    end_date: date,
    run_id: str,
) -> tuple[list[Row], dict[str, int]]:
    ingested_at = datetime.now(timezone.utc)
    expected_quotes = set(quote_currencies)
    per_currency_rows_fetched = Counter()
    rows = []

    reader = csv.DictReader(StringIO(csv_text))
    for raw_row in reader:
        if (raw_row.get("FREQ") or "").strip() != "D":
            continue

        quote_currency = (raw_row.get("CURRENCY") or "").strip().upper()
        base_currency = (raw_row.get("CURRENCY_DENOM") or "").strip().upper()
        time_period = (raw_row.get("TIME_PERIOD") or "").strip()
        obs_value = (raw_row.get("OBS_VALUE") or "").strip()

        if quote_currency not in expected_quotes or base_currency != "EUR":
            continue
        if not time_period or not obs_value:
            continue

        rate_date = parse_iso_date(time_period, "TIME_PERIOD")
        if rate_date < start_date or rate_date > end_date:
            continue

        payload_hash = hashlib.sha256(
            json.dumps(raw_row, sort_keys=True, separators=(",", ":")).encode("utf-8")
        ).hexdigest()

        rows.append(
            Row(
                base_currency=base_currency,
                quote_currency=quote_currency,
                rate_date=rate_date,
                rate=float(obs_value),
                ingested_at=ingested_at,
                run_id=run_id,
                payload_hash=payload_hash,
            )
        )
        per_currency_rows_fetched[quote_currency] += 1

    return rows, dict(sorted(per_currency_rows_fetched.items()))


dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("quote_currencies", "USD")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "7")
dbutils.widgets.text("run_id", str(uuid.uuid4()))

CATALOG = dbutils.widgets.get("catalog").strip() or "market_macro"
QUOTE_CURRENCIES = parse_quote_currencies(dbutils.widgets.get("quote_currencies").strip())
MODE = dbutils.widgets.get("mode").strip().lower()
RUN_ID = dbutils.widgets.get("run_id").strip()
TARGET_TABLE = f"{CATALOG}.brz_macro.raw_ecb_fx_ref_rates_daily"

START_DATE, END_DATE = resolve_date_window(
    mode=MODE,
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
)

if not spark.catalog.tableExists(TARGET_TABLE):
    raise RuntimeError(
        f"Target table {TARGET_TABLE} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
    )

print(
    f"Fetching ECB FX reference rates for {','.join(QUOTE_CURRENCIES)} from {START_DATE.isoformat()} "
    f"to {END_DATE.isoformat()} in {MODE} mode"
)

REQUEST_URL, SERIES_KEY, CSV_TEXT = fetch_ecb_csv(
    quote_currencies=QUOTE_CURRENCIES,
    start_date=START_DATE,
    end_date=END_DATE,
)

rows, per_currency_rows_fetched = parse_ecb_rows(
    csv_text=CSV_TEXT,
    quote_currencies=QUOTE_CURRENCIES,
    start_date=START_DATE,
    end_date=END_DATE,
    run_id=RUN_ID,
)

rows_fetched = len(rows)
print(
    f"ECB: fetched={rows_fetched} currencies={len(per_currency_rows_fetched)} series_key={SERIES_KEY}"
)

if not rows:
    result = {
        "status": "success_empty",
        "mode": MODE,
        "quote_currencies": QUOTE_CURRENCIES,
        "start_date": START_DATE.isoformat(),
        "end_date": END_DATE.isoformat(),
        "request_url": REQUEST_URL,
        "series_key": SERIES_KEY,
        "rows_fetched": 0,
        "rows_after_dedup": 0,
        "rows_to_update": 0,
        "rows_to_insert": 0,
        "rows_merged": 0,
        "run_id": RUN_ID,
        "target_table": TARGET_TABLE,
        "per_currency_rows_fetched": {},
    }
else:
    schema = StructType(
        [
            StructField("base_currency", StringType(), False),
            StructField("quote_currency", StringType(), False),
            StructField("rate_date", DateType(), False),
            StructField("rate", DoubleType(), True),
            StructField("ingested_at", TimestampType(), False),
            StructField("run_id", StringType(), False),
            StructField("payload_hash", StringType(), True),
        ]
    )

    raw_df = spark.createDataFrame(rows, schema=schema)
    dedup_window = Window.partitionBy("base_currency", "quote_currency", "rate_date").orderBy(F.col("payload_hash").desc())
    deduped_df = (
        raw_df.withColumn("_row_number", F.row_number().over(dedup_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    rows_after_dedup = deduped_df.count()
    display(deduped_df.orderBy("quote_currency", "rate_date"))

    existing_keys_df = (
        spark.table(TARGET_TABLE)
        .select("base_currency", "quote_currency", "rate_date")
        .dropDuplicates()
        .withColumn("_exists", F.lit(1))
    )
    staged_df = deduped_df.alias("src").join(
        existing_keys_df.alias("tgt"),
        on=["base_currency", "quote_currency", "rate_date"],
        how="left",
    )
    rows_to_update = staged_df.filter(F.col("_exists").isNotNull()).count()
    rows_to_insert = staged_df.filter(F.col("_exists").isNull()).count()

    DeltaTable.forName(spark, TARGET_TABLE).alias("tgt").merge(
        deduped_df.alias("src"),
        "tgt.base_currency = src.base_currency AND tgt.quote_currency = src.quote_currency AND tgt.rate_date = src.rate_date",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    result = {
        "status": "success",
        "mode": MODE,
        "quote_currencies": QUOTE_CURRENCIES,
        "start_date": START_DATE.isoformat(),
        "end_date": END_DATE.isoformat(),
        "request_url": REQUEST_URL,
        "series_key": SERIES_KEY,
        "rows_fetched": rows_fetched,
        "rows_after_dedup": rows_after_dedup,
        "rows_to_update": rows_to_update,
        "rows_to_insert": rows_to_insert,
        "rows_merged": rows_after_dedup,
        "run_id": RUN_ID,
        "target_table": TARGET_TABLE,
        "per_currency_rows_fetched": per_currency_rows_fetched,
    }

result